# Практика. Векторные вычисления

In [ ]:
import numpy as np
import pandas as pd
import time

Настоящее занятие направлено на решение ключевой проблемы оптимизации вычислительных процессов: перехода от последовательного (итерационного) к параллельному (векторному) стилю мышления. В рамках этого мы:

**Устраним императивный подход:** убедимся, что использование циклов для операций над элементами данных является архаичным и неэффективным методом, и изучим синтаксис и семантику их векторных альтернатив.

**Освоим аппарат условной селекции:** исследуем булевы маски как основной инструмент декларативного описания условий для фильтрации, уделив особое внимание комбинированию простых предикатов в сложные логические выражения.

>#### Задание 1
Загрузите датасет retail из папки data. Просмотрите информацию о данных при помощи метода `info()`

In [2]:
df = pd.read_csv('data/retail.csv')
display(df)
df.info()

,transaction_id,product_category,quantity,is_member,hour_of_day,unit_price
0,T047283,Clothing,20,1,15,439.93
1,T093646,Electronics,10,0,16,1176.39
2,T010669,Groceries,18,1,21,342.14
3,T096743,Electronics,5,0,10,3771.35
4,T043460,Clothing,11,0,20,1415.86
...,...,...,...,...,...,...
99995,T085954,Home,7,1,9,1256.16
99996,T087063,Home,11,0,17,889.21
99997,T046509,Books,6,0,15,527.84
99998,T096124,Home,12,1,19,1190.78


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    100000 non-null  object 
 1   product_category  100000 non-null  object 
 2   quantity          100000 non-null  int64  
 3   is_member         100000 non-null  int64  
 4   hour_of_day       100000 non-null  int64  
 5   unit_price        100000 non-null  float64
dtypes: float64(1), int64(3), object(2)
memory usage: 4.6+ MB


Данные одержат сведения о транзакциях в розничном магазине:
- transaction_id — уникальный идентификатор транзакции (строка, например 'T000001'),
- product_category — категория товара: 'Electronics', 'Clothing', 'Groceries', 'Home', 'Books',
- quantity — количество товара в транзакции (целое число, от 1 до 20),
- unit_price — цена за единицу товара в рублях (число с плавающей точкой, от 50 до 5000),
- is_member — флаг членства в программе лояльности (0 — нет, 1 — да),
- hour_of_day — час дня, когда совершена покупка (целое число, от 9 до 21 — рабочие часы магазина).

>#### Задание 2
>Расчитайте суммарную выручку:
>1. итерационно, при помощи циклов,
>2. при помощи векторных операций.
>
>Измерьте время исполнения, сравните скорость.

In [3]:
start_time_for_loop = time.time()

total = 0

for i in range(len(df)):
    total = total + df['unit_price'][i] * df['quantity'][i]

time_for_loop = time.time() - start_time_for_loop

print('total =', round(total, 2))
print('time_for_loop =', time_for_loop)

total = 1317235305.42
time_for_loop = 0.5730018615722656


In [4]:
start_time_vector = time.time()

total = (df['unit_price'] * df['quantity']).sum()

time_vector = time.time() - start_time_vector
if time_vector == 0:
    time_vector = 0.0000005

print('total =', round(total, 2))
print('time_vector =', time_vector)

total = 1317235305.42
time_vector = 0.00099945068359375


In [5]:
speed_up = round(time_for_loop / time_vector)

print('speed_up =', speed_up)

speed_up = 573


>#### Задание 3
Будем считать, что транзакция относится к категории Premium, если одновременно выполняются два условия: 1) сумма транзакции превышает 99000, 2) приобретаются товары из категории Electronics или Home. Найдите премиальные транзакции:
>1. итерационно, при помощи циклов,
>2. при помощи построения сложной булевой маски.
>
>Измерьте время исполнения, сравните скорость

In [6]:
start_time_for_loop = time.time()

premium = []

for i in range(len(df)):
    is_expensive = df['unit_price'][i] * df['quantity'][i] > 99000
    is_target_category = df['product_category'][i] == 'Electronics' or df['product_category'][i] == 'Home'
    if is_expensive and is_target_category:
        premium.append(df['transaction_id'][i])

time_for_loop = time.time() - start_time_for_loop

print('premium =', premium)
print('time_for_loop =', time_for_loop)

premium = ['T076983', 'T006588', 'T048118', 'T057927', 'T089571', 'T021749', 'T076463', 'T081658', 'T062328', 'T000959', 'T010936', 'T082661', 'T014331']
time_for_loop = 1.0519733428955078


In [7]:
start_time_vector = time.time()

revenue = df['unit_price'] * df['quantity']
is_expensive_mask = revenue > 99000
is_target_category_mask = (df['product_category'] == 'Electronics') | (df['product_category'] == 'Home')

mask = is_expensive_mask & is_target_category_mask
premium = df['transaction_id'][mask].values

time_vector = time.time() - start_time_vector
if time_vector == 0:
    time_vector = 0.0000005

print('premium =', premium)
print('time_vector =', time_vector)

premium = ['T076983' 'T006588' 'T048118' 'T057927' 'T089571' 'T021749' 'T076463'
 'T081658' 'T062328' 'T000959' 'T010936' 'T082661' 'T014331']
time_vector = 0.009618282318115234


In [8]:
speed_up = round(time_for_loop / time_vector)

print('speed_up =', speed_up)

speed_up = 109


>#### Задание 4
Найдите среднюю сумму транзакции для участников программы лояльности и для всех остальных покупателей. Выполните это задание:
>1. итерационно, при помощи циклов,
>2. при помощи векторных операций.
>
>Измерьте время исполнения, сравните скорость.

In [9]:
start_time_for_loop = time.time()

sum_member = 0
count_member = 0
sum_not_member = 0
count_not_member = 0

for i in range(len(df)):
    amount = df['unit_price'][i] * df['quantity'][i]
    if df['is_member'][i] == 1:
        sum_member = sum_member + amount
        count_member = count_member + 1
    else:
        sum_not_member = sum_not_member + amount
        count_not_member = count_not_member + 1

average_member = round(sum_member / count_member, 2)
average_not_member = round(sum_not_member / count_not_member, 2)

time_for_loop = time.time() - start_time_for_loop

print('average_member =', average_member)
print('average_not_member =', average_not_member)
print('time_for_loop =', time_for_loop)

average_member = 13095.4
average_not_member = 13223.55
time_for_loop = 0.8423051834106445


In [10]:
start_time_vector = time.time()

amount = df['unit_price'] * df['quantity']
member_mask = df['is_member'] == 1

average_member = round(amount[member_mask].mean(), 2)
average_not_member = round(amount[~member_mask].mean(), 2)

time_vector = time.time() - start_time_vector
if time_vector == 0:
    time_vector = 0.0000005

print('average_member =', average_member)
print('average_not_member =', average_not_member)
print('time_vector =', time_vector)

average_member = 13095.4
average_not_member = 13223.55
time_vector = 0.0029997825622558594


In [11]:
speed_up = round(time_for_loop / time_vector)

print('speed_up =', speed_up)

speed_up = 281


>#### Задание 5
Определите, в какие часы участники программы лояльности совершают больше всего покупок (по количеству транзакций). Выполните это задание:
>1. итерационно, при помощи циклов,
>2. при помощи векторных операций.
>
>Измерьте время исполнения, сравните скорость.

In [12]:
start_time_for_loop = time.time()

hours = range(9, 22)
counts_by_hour = {hour: 0 for hour in hours}

for i in range(len(df)):
    if df['is_member'][i] == 1:
        hour = df['hour_of_day'][i]
        counts_by_hour[hour] = counts_by_hour[hour] + 1

hour_max_transactions = max(counts_by_hour, key=counts_by_hour.get)

time_for_loop = time.time() - start_time_for_loop

print('hour_max_transactions =', hour_max_transactions)
print('time_for_loop =', time_for_loop)

hour_max_transactions = 16
time_for_loop = 0.3861813545227051


In [13]:
start_time_vector = time.time()

member_mask = df['is_member'] == 1
hour_max_transactions = df['hour_of_day'][member_mask].value_counts().idxmax()

time_vector = time.time() - start_time_vector
if time_vector == 0:
    time_vector = 0.0000005

print('hour_max_transactions =', hour_max_transactions)
print('time_vector =', time_vector)

hour_max_transactions = 16
time_vector = 0.004013538360595703


In [14]:
speed_up = round(time_for_loop / time_vector)

print('speed_up =', speed_up)

speed_up = 96


# Домашнее задание

>#### Задание
Прочитайте файл server_logs.csv из папки data.

In [15]:
df = pd.read_csv('data/server_logs.csv')
display(df)
df.info()

,request_id,endpoint,response_time_ms,status_code,client_type,request_size_kb
0,REQ-066148,/auth/logout,824,200,mobile_app,450.786439
1,REQ-080374,/auth/logout,2823,400,mobile_app,216.305318
2,REQ-038778,/orders,1667,200,mobile_app,298.607141
3,REQ-104788,/orders,2019,401,browser,687.819079
4,REQ-149255,/auth/login,703,200,mobile_app,459.176544
...,...,...,...,...,...,...
149995,REQ-052010,/users,4814,200,script,874.658671
149996,REQ-131042,/users,3134,400,browser,208.283995
149997,REQ-096289,/orders,73,400,script,189.767642
149998,REQ-056748,/products,761,200,browser,497.939931


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   request_id        150000 non-null  object 
 1   endpoint          150000 non-null  object 
 2   response_time_ms  150000 non-null  int64  
 3   status_code       150000 non-null  int64  
 4   client_type       150000 non-null  object 
 5   request_size_kb   150000 non-null  float64
dtypes: float64(1), int64(2), object(3)
memory usage: 6.9+ MB


Данные содержат записи о HTTP-запросах к API веб-сервиса за сутки. Структура данных:
- request_id — уникальный идентификатор запроса (строка, например 'REQ-000001'),
- endpoint — конечная точка API: '/users', '/orders', '/products', '/auth/login', '/auth/logout',
- response_time_ms — время обработки запроса сервером в миллисекундах (целое число, от 20 до 5000),
- status_code — HTTP-статус ответа (целое число, например 200, 201, 400, 401, 404, 500),
- client_type — тип клиента: 'browser', 'mobile_app', 'script', 'other',
- request_size_kb — размер тела запроса в килобайтах (число с плавающей точкой, от 0.1 до 1024.0).

>#### Задание
>1. Рассчитайте нагрузку и эффективность:
>- добавьте новый столбец is_slow (булев), который равен True, если время отклика превышает 1000 мс (1 секунду),
>- добавьте новый столбец is_large_request (булев), который равен True, если размер запроса превышает 500 КБ.
>2. Выделите критические запросы по двум независимым критериям:
>- критерий A — запрос завершился с ошибкой клиента или сервера (status_code >= 400),
>- критерий B — запрос является одновременно медленным (is_slow == True) и большим (is_large_request == True),
>- запрос считается критическим, если выполняется хотя бы один из этих критериев.
>3. Проанализируйте поведение разных типов клиентов:
>- найдите среднее время отклика отдельно для запросов от 'mobile_app' и для всех остальных типов клиентов.
>- определите, есть ли статусы ошибок (status_code >= 400) среди запросов, отправленных из 'browser'.

Выполните все задания только в векторной манере, не прибегая к циклам.

In [16]:
# 1. Нагрузка и эффективность
df['is_slow'] = df['response_time_ms'] > 1000
df['is_large_request'] = df['request_size_kb'] > 500

display(df[['response_time_ms', 'is_slow', 'request_size_kb', 'is_large_request']].head())

# 2. Критические запросы по двум критериям
criterion_a = df['status_code'] >= 400
criterion_b = df['is_slow'] & df['is_large_request']

df['is_critical'] = criterion_a | criterion_b

critical_requests = df['request_id'][df['is_critical']]
print('Число критических запросов:', len(critical_requests))
display(critical_requests.head())

# 3.1 Среднее время отклика: mobile_app и остальные
mobile_mask = df['client_type'] == 'mobile_app'
average_time_mobile = df['response_time_ms'][mobile_mask].mean()
average_time_not_mobile = df['response_time_ms'][~mobile_mask].mean()

print('average_time_mobile     =', round(average_time_mobile, 2))
print('average_time_not_mobile =', round(average_time_not_mobile, 2))

# 3.2 Есть ли ошибки среди запросов browser
browser_error_mask = (df['client_type'] == 'browser') & (df['status_code'] >= 400)
has_browser_errors = browser_error_mask.any()

print('has_browser_errors =', has_browser_errors)
print('Число ошибок browser =', browser_error_mask.sum())

,response_time_ms,is_slow,request_size_kb,is_large_request
0,824,False,450.786439,False
1,2823,True,216.305318,False
2,1667,True,298.607141,False
3,2019,True,687.819079,True
4,703,False,459.176544,False


Число критических запросов: 83581


1    REQ-080374
3    REQ-104788
5    REQ-036568
6    REQ-008826
7    REQ-058338
Name: request_id, dtype: object

average_time_mobile     =

 2501.6
average_time_not_mobile = 2511.27


has_browser_errors = True
Число ошибок browser = 14843
